# Formula 1 2023 Season — Full Analytics Pipeline
## MSBA305 · Data Processing Frameworks · American University of Beirut
**Authors:** Aya Masri · Carmen Fhaily · Amira Merheb · Karim Chaar · Laure Awar  
**Date:** April 2026

---

## Notebook Structure

| Part | Title | Description |
|------|-------|-------------|
| **1** | Data Loading, Inspection & Cleaning | Load CSV, JSON, XML sources → inspect → clean → build `df_master` |
| **2** | SQL Analysis | 5 SQL queries on `df_master` using SQLite (aggregation, window functions, CTEs) |
| **3** | Interactive Dashboard | 6 Plotly charts with interpretations answering the 5 research questions |

### Research Questions
1. Which drivers and constructors accumulated the most points across the 2023 season, and how did performance evolve race by race?
2. Is there a relationship between a driver's starting grid position and their final race result?
3. Which circuits produced the most competitive races (smallest gap between finishers)?
4. How do circuit characteristics (length, location) correlate with race outcomes?
5. Which teams dominated specific circuit types, and how did reliability differ across teams?


---
# Part 1 · Data Loading, Inspection & Cleaning
Three raw data sources are loaded, inspected for quality issues, individually cleaned, then joined into a single master dataset `df_master`.


## 1.1 · Imports

In [1]:
import pandas as pd
import json
import xml.etree.ElementTree as ET
import os
import sqlite3
import warnings
warnings.filterwarnings('ignore')

DATA_ROOT = './'   # all data files must be in the same folder as this notebook
print("DATA_ROOT set to", DATA_ROOT)


DATA_ROOT set to ./


## 1.2 · Loading the Three Data Sources

In [2]:
# ── CSV: 2023 Race Results ──────────────────────────────────────────────────
csv_path = DATA_ROOT + 'Formula1_2023season_raceResults.csv'
df_csv = pd.read_csv(csv_path)
print(f"CSV loaded: {df_csv.shape[0]} rows × {df_csv.shape[1]} columns")
df_csv.head(3)


CSV loaded: 440 rows × 11 columns


,Track,Position,No,Driver,Team,Starting Grid,Laps,Time/Retired,Points,Set Fastest Lap,Fastest Lap Time
0,Bahrain,1,1,Max Verstappen,Red Bull Racing Honda RBPT,1,57,1:33:56.736,25,No,1:36.236
1,Bahrain,2,11,Sergio Perez,Red Bull Racing Honda RBPT,2,57,+11.987,18,No,1:36.344
2,Bahrain,3,14,Fernando Alonso,Aston Martin Aramco Mercedes,5,57,+38.637,15,No,1:36.156


In [3]:
# ── JSON: Ergast API race results dump ──────────────────────────────────────
json_path = DATA_ROOT + 'JSON-F1-full (1).json'
with open(json_path) as f:
    json_raw = json.load(f)
races = json_raw['MRData']['RaceTable']['Races']
print(f"JSON loaded: {len(races)} races")
print(f"First race : {races[0]['raceName']}  |  Results in first race: {len(races[0]['Results'])}")


JSON loaded: 29 races
First race : Bahrain Grand Prix  |  Results in first race: 20


In [4]:
# ── XML: Circuit specifications ─────────────────────────────────────────────
xml_path = DATA_ROOT + 'f1_circuits.xml'
tree = ET.parse(xml_path)
root = tree.getroot()
circuits = root.findall('circuit')
print(f"XML loaded: {len(circuits)} circuits")
print("Sample fields:", [field.tag for field in circuits[0]])


XML loaded: 40 circuits
Sample fields: ['circuitId', 'name', 'country', 'city', 'length_km', 'opened', 'longitude', 'latitude']


## 1.3 · Data Inspection

In [5]:
# ── CSV inspection ──────────────────────────────────────────────────────────
print("=== CSV shape & dtypes ===")
print(df_csv.shape)
print(df_csv.dtypes)
print("\nNull counts:")
print(df_csv.isnull().sum())


=== CSV shape & dtypes ===
(440, 11)
Track                 str
Position              str
No                  int64
Driver                str
Team                  str
Starting Grid       int64
Laps                int64
Time/Retired          str
Points              int64
Set Fastest Lap       str
Fastest Lap Time      str
dtype: object

Null counts:
Track                0
Position             0
No                   0
Driver               0
Team                 0
Starting Grid        0
Laps                 0
Time/Retired         0
Points               0
Set Fastest Lap      0
Fastest Lap Time    13
dtype: int64


**Findings — CSV:**  
- `Position` is `object` (string) because of `NC` (Not Classified) and `DQ` (Disqualified) values.  
- 13 nulls in `Fastest Lap Time` — all for NC drivers (expected).  
- `Time/Retired` is mixed format: winner total time, gap to winner (`+Xs`), lapped (`+N laps`), or `DNF`/`DNS`.


In [6]:
# ── JSON inspection ─────────────────────────────────────────────────────────
race   = races[0]
result = race['Results'][0]
print("Race-level fields  :", list(race.keys()))
print("Result-level fields:", list(result.keys()))
print("Driver fields      :", list(result['Driver'].keys()))
print("FastestLap fields  :", result.get('FastestLap'))


Race-level fields  : ['season', 'round', 'url', 'raceName', 'Circuit', 'date', 'time', 'Results']
Result-level fields: ['number', 'position', 'positionText', 'points', 'Driver', 'Constructor', 'grid', 'laps', 'status', 'Time', 'FastestLap']
Driver fields      : ['driverId', 'permanentNumber', 'code', 'url', 'givenName', 'familyName', 'dateOfBirth', 'nationality']
FastestLap fields  : {'rank': '6', 'lap': '44', 'Time': {'time': '1:36.236'}, 'AverageSpeed': {'units': 'kph', 'speed': '202.452'}}


**Findings — JSON:**  
- 29 race objects but CSV has 22 — 7 extras are sprint races (excluded from main results table).  
- All fields are strings; numeric and date casting needed.  
- Heavily nested: driver info and constructor info are sub-dicts inside each result.


In [7]:
# ── XML inspection ──────────────────────────────────────────────────────────
missing = {}
for circuit in circuits:
    for field in circuit:
        missing.setdefault(field.tag, 0)
        if field.text is None or field.text.strip() == '':
            missing[field.tag] += 1
print(f"Total circuits: {len(circuits)}")
print("Missing values per field:", missing)


Total circuits: 40
Missing values per field: {'circuitId': 40, 'name': 0, 'country': 40, 'city': 40, 'length_km': 0, 'opened': 0, 'longitude': 0, 'latitude': 0}


**Findings — XML:**  
- `circuitId`, `country`, and `city` are completely empty across all 40 rows — these columns will be dropped.  
- `length_km`, `opened`, `longitude`, `latitude` are fully populated and usable.


## 1.4 · Data Cleaning

### CSV Cleaning

In [8]:
# Non-numeric Position values (NC / DQ)
print("Non-numeric positions found:")
print(df_csv[~df_csv['Position'].str.isnumeric()][['Track','Driver','Position']]
      .groupby('Position').size().reset_index(name='count'))


Non-numeric positions found:
  Position  count
0       DQ      2
1       NC     50


In [9]:
# Preserve NC/DQ status, then cast Position to numeric
df_csv['position_status'] = df_csv['Position'].apply(
    lambda x: 'NC' if x == 'NC' else ('DQ' if x == 'DQ' else 'Finished')
)
df_csv['Position'] = pd.to_numeric(df_csv['Position'], errors='coerce')

print("position_status counts:")
print(df_csv['position_status'].value_counts())
print(f"\nPosition nulls after cast: {df_csv['Position'].isnull().sum()}")


position_status counts:
position_status
Finished    388
NC           50
DQ            2
Name: count, dtype: int64

Position nulls after cast: 52


In [10]:
# Categorise Time/Retired column
def classify_time(val):
    if pd.isnull(val):              return 'no_time'
    if val.startswith('+') and 'lap' in val: return 'lapped'
    if val.startswith('+'):         return 'gap_to_winner'
    if val in ['DNF', 'DNS']:       return 'dnf_dns'
    return 'winner_time'

df_csv['time_type'] = df_csv['Time/Retired'].apply(classify_time)
print("Time/Retired categories:")
print(df_csv['time_type'].value_counts())
print(f"Total: {len(df_csv)} ✓")


Time/Retired categories:
time_type
gap_to_winner    284
lapped            70
dnf_dns           64
winner_time       22
Name: count, dtype: int64
Total: 440 ✓


### JSON Cleaning

In [11]:
# Flatten nested JSON into a flat DataFrame
rows = []
for race in races:
    for result in race['Results']:
        rows.append({
            'season':       race['season'],
            'round':        race['round'],
            'race_name':    race['raceName'],
            'race_date':    race['date'],
            'driver_id':    result['Driver']['driverId'],
            'driver_code':  result['Driver']['code'],
            'driver_name':  result['Driver']['givenName'] + ' ' + result['Driver']['familyName'],
            'nationality':  result['Driver']['nationality'],
            'dob':          result['Driver']['dateOfBirth'],
            'constructor':  result['Constructor']['name'],
            'grid':         result['grid'],
            'position':     result['position'],
            'points':       result['points'],
            'laps':         result['laps'],
            'status':       result['status'],
        })

df_json = pd.DataFrame(rows)
print(f"JSON flattened: {df_json.shape}")


JSON flattened: (440, 15)


In [12]:
# Cast all columns to correct types
df_json['round']     = pd.to_numeric(df_json['round'])
df_json['grid']      = pd.to_numeric(df_json['grid'])
df_json['position']  = pd.to_numeric(df_json['position'], errors='coerce')
df_json['points']    = pd.to_numeric(df_json['points'])
df_json['laps']      = pd.to_numeric(df_json['laps'])
df_json['race_date'] = pd.to_datetime(df_json['race_date'])
df_json['dob']       = pd.to_datetime(df_json['dob'])

# Fix driver name mismatches between CSV and JSON
name_fix = {
    'Sergio Pérez':    'Sergio Perez',
    'Nico Hülkenberg': 'Nico Hulkenberg',
    'Nyck de Vries':   'Nyck De Vries',
}
df_json['driver_name'] = df_json['driver_name'].replace(name_fix)

print("Types after casting:")
print(df_json.dtypes)
print(f"\nUnique races: {df_json['race_name'].nunique()}  |  Entries per race: {df_json.groupby('race_name').size().unique()}")


Types after casting:
season                    str
round                   int64
race_name                 str
race_date      datetime64[us]
driver_id                 str
driver_code               str
driver_name               str
nationality               str
dob            datetime64[us]
constructor               str
grid                    int64
position                int64
points                  int64
laps                    int64
status                    str
dtype: object

Unique races: 22  |  Entries per race: [20]


### XML Cleaning

In [13]:
# Convert XML to DataFrame (keeping only populated fields)
rows = []
for circuit in circuits:
    rows.append({
        'circuit_name': circuit.find('name').text,
        'length_km':    circuit.find('length_km').text,
        'opened':       circuit.find('opened').text,
        'longitude':    circuit.find('longitude').text,
        'latitude':     circuit.find('latitude').text,
    })

df_xml = pd.DataFrame(rows)
df_xml['length_km'] = pd.to_numeric(df_xml['length_km'])
df_xml['opened']    = pd.to_numeric(df_xml['opened'])
df_xml['longitude'] = pd.to_numeric(df_xml['longitude'])
df_xml['latitude']  = pd.to_numeric(df_xml['latitude'])

print(f"XML DataFrame: {df_xml.shape}")
df_xml.head(5)


XML DataFrame: (40, 5)


,circuit_name,length_km,opened,longitude,latitude
0,Albert Park Circuit,5278,1953,144.968644,-37.849757
1,Bahrain International Circuit,5412,2002,50.510539,26.031766
2,Shanghai International Circuit,5451,2004,121.218380,31.336835
3,Circuit de Barcelona-Catalunya,4655,1991,2.261221,41.570034
4,Circuit de Monaco,3337,1929,7.427191,43.739404


## 1.5 · Building df_master — Joining All Three Sources

In [14]:
# Map CSV track names → JSON race names & XML circuit names
csv_to_json = {
    'Abu Dhabi':'Abu Dhabi Grand Prix','Australia':'Australian Grand Prix',
    'Austria':'Austrian Grand Prix','Azerbaijan':'Azerbaijan Grand Prix',
    'Bahrain':'Bahrain Grand Prix','Belgium':'Belgian Grand Prix',
    'Brazil':'São Paulo Grand Prix','Canada':'Canadian Grand Prix',
    'Great Britain':'British Grand Prix','Hungary':'Hungarian Grand Prix',
    'Italy':'Italian Grand Prix','Japan':'Japanese Grand Prix',
    'Las Vegas':'Las Vegas Grand Prix','Mexico':'Mexico City Grand Prix',
    'Miami':'Miami Grand Prix','Monaco':'Monaco Grand Prix',
    'Netherlands':'Dutch Grand Prix','Qatar':'Qatar Grand Prix',
    'Saudi Arabia':'Saudi Arabian Grand Prix','Singapore':'Singapore Grand Prix',
    'Spain':'Spanish Grand Prix','United States':'United States Grand Prix',
}
csv_to_xml = {
    'Abu Dhabi':'Yas Marina Circuit','Australia':'Albert Park Circuit',
    'Austria':'Red Bull Ring','Azerbaijan':'Baku City Circuit',
    'Bahrain':'Bahrain International Circuit','Belgium':'Circuit de Spa-Francorchamps',
    'Brazil':'Autódromo José Carlos Pace - Interlagos','Canada':'Circuit Gilles-Villeneuve',
    'Great Britain':'Silverstone Circuit','Hungary':'Hungaroring',
    'Italy':'Autodromo Nazionale Monza','Japan':'Suzuka International Racing Course',
    'Las Vegas':'Las Vegas Street Circuit','Mexico':'Autódromo Hermanos Rodríguez',
    'Miami':'Miami International Autodrome','Monaco':'Circuit de Monaco',
    'Netherlands':'Circuit Zandvoort','Qatar':'Losail International Circuit',
    'Saudi Arabia':'Jeddah Corniche Circuit','Singapore':'Marina Bay Street Circuit',
    'Spain':'Circuit de Barcelona-Catalunya','United States':'Circuit of the Americas',
}

df_csv['race_name']    = df_csv['Track'].map(csv_to_json)
df_csv['circuit_name'] = df_csv['Track'].map(csv_to_xml)
print(f"Unmapped race_names:    {df_csv['race_name'].isnull().sum()}")
print(f"Unmapped circuit_names: {df_csv['circuit_name'].isnull().sum()}")


Unmapped race_names:    0
Unmapped circuit_names: 0


In [15]:
# Merge CSV ← JSON (driver nationality, DOB, status, driver_code, driver_id)
df_master = df_csv.merge(
    df_json[['race_name','driver_name','nationality','dob','driver_code','driver_id','status']],
    left_on=['race_name','Driver'], right_on=['race_name','driver_name'], how='left'
)

# Merge ← XML (circuit specs)
df_master = df_master.merge(df_xml, on='circuit_name', how='left')

# Drop duplicate column
df_master = df_master.drop(columns=['driver_name'])

print(f"After join: {df_master.shape}")
print(f"Null counts:\n{df_master.isnull().sum()[df_master.isnull().sum()>0]}")


After join: (440, 24)
Null counts:
Position            52
Fastest Lap Time    13
dtype: int64


In [16]:
# Rename to snake_case
df_master = df_master.rename(columns={
    'Track':'track','Position':'finish_position','No':'driver_number',
    'Driver':'driver_name','Team':'team','Starting Grid':'grid_position',
    'Laps':'laps','Time/Retired':'time_retired','Points':'points',
    'Set Fastest Lap':'set_fastest_lap','Fastest Lap Time':'fastest_lap_time',
})

# Add race_date from JSON mapping
race_date_map = df_json[['race_name','race_date']].drop_duplicates()
df_master = df_master.merge(race_date_map, on='race_name', how='left')
df_master = df_master.drop(columns=['race_name','circuit_name'])

# Reorder columns
df_master = df_master[[
    'race_date','track','driver_name','driver_code','driver_id','driver_number',
    'nationality','dob','team','grid_position','finish_position','position_status',
    'laps','time_retired','time_type','points','set_fastest_lap','fastest_lap_time',
    'status','length_km','opened','longitude','latitude',
]]

print(f"df_master final shape: {df_master.shape}")
print("Columns:", df_master.columns.tolist())
df_master.head(3)


df_master final shape: (440, 23)
Columns: ['race_date', 'track', 'driver_name', 'driver_code', 'driver_id', 'driver_number', 'nationality', 'dob', 'team', 'grid_position', 'finish_position', 'position_status', 'laps', 'time_retired', 'time_type', 'points', 'set_fastest_lap', 'fastest_lap_time', 'status', 'length_km', 'opened', 'longitude', 'latitude']


,race_date,track,driver_name,driver_code,driver_id,driver_number,nationality,dob,team,grid_position,...,time_retired,time_type,points,set_fastest_lap,fastest_lap_time,status,length_km,opened,longitude,latitude
0,2023-03-05,Bahrain,Max Verstappen,VER,max_verstappen,1,Dutch,1997-09-30,Red Bull Racing Honda RBPT,1,...,1:33:56.736,winner_time,25,No,1:36.236,Finished,5412,2002,50.510539,26.031766
1,2023-03-05,Bahrain,Sergio Perez,PER,perez,11,Mexican,1990-01-26,Red Bull Racing Honda RBPT,2,...,+11.987,gap_to_winner,18,No,1:36.344,Finished,5412,2002,50.510539,26.031766
2,2023-03-05,Bahrain,Fernando Alonso,ALO,alonso,14,Spanish,1981-07-29,Aston Martin Aramco Mercedes,5,...,+38.637,gap_to_winner,15,No,1:36.156,Finished,5412,2002,50.510539,26.031766


In [17]:
# Save df_master locally
df_master.to_csv('df_master.csv', index=False)
print(f"✓ df_master saved to df_master.csv")
print(f"  Shape: {df_master.shape}")
print(f"  Races: {df_master['track'].nunique()}  |  Drivers: {df_master['driver_name'].nunique()}  |  Teams: {df_master['team'].nunique()}")


✓ df_master saved to df_master.csv
  Shape: (440, 23)
  Races: 22  |  Drivers: 22  |  Teams: 10


---
# Part 2 · SQL Analysis
`df_master` is loaded into an in-memory SQLite database and queried with five progressively complex SQL statements — from simple aggregation to window functions and CTEs.


## 2.1 · Load df_master into SQLite

In [18]:
# df_master is already in memory from Part 1.
# If running this section independently, uncomment the line below:
# df_master = pd.read_csv('df_master.csv')

conn = sqlite3.connect(':memory:')
df_master.to_sql('f1', conn, index=False, if_exists='replace')
print(f"✓ Loaded into SQLite: {pd.read_sql_query('SELECT COUNT(*) AS n FROM f1', conn).iloc[0,0]} rows")


✓ Loaded into SQLite: 440 rows


## 2.2 · Query 1 — Championship Standings (Simple Aggregation)

In [19]:
q1 = pd.read_sql_query("""
    SELECT driver_name, nationality, team,
           SUM(points)  AS total_points,
           COUNT(*)     AS races_entered
    FROM   f1
    GROUP  BY driver_name, nationality, team
    ORDER  BY total_points DESC
    LIMIT  10
""", conn)

print("=== Q1: Championship Standings (Top 10) ===")
display(q1)


=== Q1: Championship Standings (Top 10) ===


,driver_name,nationality,team,total_points,races_entered
0,Max Verstappen,Dutch,Red Bull Racing Honda RBPT,530,22
1,Sergio Perez,Mexican,Red Bull Racing Honda RBPT,260,22
2,Lewis Hamilton,British,Mercedes,217,22
3,Fernando Alonso,Spanish,Aston Martin Aramco Mercedes,198,22
4,Charles Leclerc,Monegasque,Ferrari,185,22
5,Lando Norris,British,McLaren Mercedes,184,22
6,Carlos Sainz,Spanish,Ferrari,178,22
7,George Russell,British,Mercedes,157,22
8,Oscar Piastri,Australian,McLaren Mercedes,82,22
9,Lance Stroll,Canadian,Aston Martin Aramco Mercedes,68,22


**Finding:** Max Verstappen leads with 530 points — 270 ahead of team-mate Sergio Pérez (260). Red Bull's combined 790 points dwarfs every other constructor. Hamilton (217), Alonso (198), and Leclerc (185) are clustered closely for positions 3–5.


## 2.3 · Query 2 — DNF Rate per Team (Filtering + Aggregation)

In [20]:
q2 = pd.read_sql_query("""
    SELECT team,
           COUNT(*)  AS total_entries,
           SUM(CASE WHEN status != 'Finished' THEN 1 ELSE 0 END) AS dnfs,
           ROUND(100.0 * SUM(CASE WHEN status != 'Finished' THEN 1 ELSE 0 END)
                       / COUNT(*), 1) AS dnf_pct
    FROM   f1
    GROUP  BY team
    ORDER  BY dnf_pct DESC
""", conn)

print("=== Q2: DNF Rate by Team ===")
display(q2)


=== Q2: DNF Rate by Team ===


,team,total_entries,dnfs,dnf_pct
0,Haas Ferrari,44,27,61.4
1,Williams Mercedes,44,20,45.5
2,AlphaTauri Honda RBPT,44,20,45.5
3,Alfa Romeo Ferrari,44,20,45.5
4,McLaren Mercedes,44,12,27.3
5,Alpine Renault,44,12,27.3
6,Ferrari,44,8,18.2
7,Aston Martin Aramco Mercedes,44,8,18.2
8,Mercedes,44,6,13.6
9,Red Bull Racing Honda RBPT,44,3,6.8


**Finding:** Haas Ferrari suffered a 61.4% DNF rate — the worst reliability on the grid by a large margin. Alfa Romeo, AlphaTauri, and Williams all shared a 45.5% rate. Red Bull Racing had the lowest DNF rate at just 6.8% (3 non-finishes in 44 starts), reinforcing their championship dominance through both pace and reliability.


## 2.4 · Query 3 — Pole-to-Win Conversion (Multi-Condition Logic)

In [21]:
q3 = pd.read_sql_query("""
    SELECT team,
           SUM(CASE WHEN grid_position  = 1 THEN 1 ELSE 0 END) AS poles,
           SUM(CASE WHEN finish_position = 1 THEN 1 ELSE 0 END) AS wins,
           SUM(CASE WHEN grid_position = 1 AND finish_position = 1 THEN 1 ELSE 0 END) AS pole_to_win,
           ROUND(100.0
               * SUM(CASE WHEN grid_position = 1 AND finish_position = 1 THEN 1 ELSE 0 END)
               / NULLIF(SUM(CASE WHEN grid_position = 1 THEN 1 ELSE 0 END), 0), 1) AS conversion_pct
    FROM   f1
    GROUP  BY team
    HAVING poles > 0
    ORDER  BY conversion_pct DESC
""", conn)

print("=== Q3: Pole-to-Win Conversion by Team ===")
display(q3)


=== Q3: Pole-to-Win Conversion by Team ===


,team,poles,wins,pole_to_win,conversion_pct
0,Red Bull Racing Honda RBPT,14,21,13,92.9
1,Ferrari,7,1,1,14.3
2,Mercedes,1,0,0,0.0


**Finding:** Red Bull converted 13 of 14 pole positions into victories — a 92.9% conversion rate, the highest in a single season in F1 history. Ferrari took 7 poles but converted only 1 (14.3%), reflecting the gap in race-pace and strategy execution between the two teams. Mercedes took 1 pole (Russell at Brazil) and failed to convert it.


## 2.5 · Query 4 — Positions Gained/Lost per Driver (Window Functions)

In [22]:
q4 = pd.read_sql_query("""
    SELECT driver_name,
           ROUND(AVG(grid_position),   2) AS avg_grid,
           ROUND(AVG(finish_position), 2) AS avg_finish,
           ROUND(AVG(grid_position - finish_position), 2) AS avg_positions_gained,
           COUNT(*) AS races_completed
    FROM   f1
    WHERE  status = 'Finished'
    GROUP  BY driver_name
    HAVING races_completed >= 10
    ORDER  BY avg_positions_gained DESC
""", conn)

print("=== Q4: Average Positions Gained Per Race (min. 10 finishes) ===")
display(q4)


=== Q4: Average Positions Gained Per Race (min. 10 finishes) ===


,driver_name,avg_grid,avg_finish,avg_positions_gained,races_completed
0,Sergio Perez,9.16,3.89,5.26,19
1,Lance Stroll,12.00,8.81,3.19,16
2,Yuki Tsunoda,14.00,11.21,2.79,14
3,Guanyu Zhou,15.75,13.00,2.75,12
4,Esteban Ocon,11.00,8.54,2.46,13
5,Max Verstappen,3.18,1.27,1.91,22
6,Lewis Hamilton,6.40,4.90,1.50,20
7,Pierre Gasly,11.11,9.68,1.42,19
8,Valtteri Bottas,13.17,11.83,1.33,12
9,Lando Norris,7.56,6.28,1.28,18


**Finding (RQ2):** Sergio Pérez gained an average of **+5.26 positions** per finished race — the best overtaking record on the grid, frequently starting mid-pack due to grid penalties and recovering to podiums. Lance Stroll (+3.19) and Yuki Tsunoda (+2.79) also showed strong racecraft relative to their grid positions. Carlos Sainz (−0.68) and Nico Hülkenberg (−0.50) tended to lose positions from their qualifying slot, suggesting qualifying over-performance relative to race pace.


## 2.6 · Query 5 — Points Progression using CTEs + Window Functions (Most Complex)

In [23]:
q5 = pd.read_sql_query("""
    WITH cumulative_points_calc AS (
        SELECT driver_name,
               race_date,
               track,
               points,
               SUM(points) OVER (
                   PARTITION BY driver_name
                   ORDER BY race_date
                   ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
               ) AS cumulative_points
        FROM   f1
    ),
    ranked_cumulative AS (
        SELECT driver_name, race_date, track, points, cumulative_points,
               RANK() OVER (
                   PARTITION BY race_date
                   ORDER BY cumulative_points DESC
               ) AS championship_rank
        FROM   cumulative_points_calc
    )
    SELECT driver_name, track, race_date, points, cumulative_points, championship_rank
    FROM   ranked_cumulative
    WHERE  driver_name IN ('Max Verstappen','Lewis Hamilton','Fernando Alonso','Sergio Perez')
    ORDER  BY race_date, championship_rank
""", conn)

print(f"=== Q5: Cumulative Championship Progression (Top 4 Drivers) — {len(q5)} rows ===")
display(q5.head(20))


=== Q5: Cumulative Championship Progression (Top 4 Drivers) — 88 rows ===


,driver_name,track,race_date,points,cumulative_points,championship_rank
0,Max Verstappen,Bahrain,2023-03-05 00:00:00,25,25,1
1,Sergio Perez,Bahrain,2023-03-05 00:00:00,18,18,2
2,Fernando Alonso,Bahrain,2023-03-05 00:00:00,15,15,3
3,Lewis Hamilton,Bahrain,2023-03-05 00:00:00,10,10,5
4,Max Verstappen,Saudi Arabia,2023-03-19 00:00:00,19,44,1
5,Sergio Perez,Saudi Arabia,2023-03-19 00:00:00,25,43,2
6,Fernando Alonso,Saudi Arabia,2023-03-19 00:00:00,15,30,3
7,Lewis Hamilton,Saudi Arabia,2023-03-19 00:00:00,10,20,4
8,Max Verstappen,Australia,2023-04-02 00:00:00,25,69,1
9,Sergio Perez,Australia,2023-04-02 00:00:00,11,54,2


**Finding (RQ1 — race-by-race evolution):** The CTE first calculates each driver's running points total after every race, then ranks all drivers by that total at each race date. The output shows Verstappen holding P1 from Round 1 through the final race. Pérez led P2 for most of the season but was overtaken by Hamilton and Alonso in the second half as his results deteriorated. The window function `RANK() OVER (PARTITION BY race_date ORDER BY cumulative_points DESC)` efficiently computes championship standings at any point in the season without requiring a self-join.


---
# Part 3 · Interactive Analytics Dashboard
Six fully interactive Plotly charts — hover, zoom, pan, and filter directly in the notebook.  
Each visualisation is followed by an interpretation answering the project's research questions.


## 3.1 · Setup — Plotly Theme & Colour Palette

In [24]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from IPython.display import display, HTML

# Cast columns (safe to run even if Part 1 already ran)
for col in ['finish_position','grid_position','laps','points']:
    df_master[col] = pd.to_numeric(df_master[col], errors='coerce')
df_master['race_date'] = pd.to_datetime(df_master['race_date'], errors='coerce')

TEAM_COLORS = {
    'Red Bull Racing Honda RBPT':   '#0600EF',
    'Mercedes':                     '#00D2BE',
    'Ferrari':                      '#DC0000',
    'McLaren Mercedes':             '#FF8700',
    'Aston Martin Aramco Mercedes': '#006F62',
    'Alpine Renault':               '#0090FF',
    'Williams Mercedes':            '#005AFF',
    'AlphaTauri Honda RBPT':        '#2B4562',
    'Alfa Romeo Ferrari':           '#900000',
    'Haas Ferrari':                 '#AAAAAA',
}
DARK_BG  = '#15151E'
CARD_BG  = '#1A1A24'
RED      = '#E10600'
WHITE    = '#FFFFFF'
SILVER   = '#C0C0C0'
GREEN    = '#00FF41'

layout_base = dict(
    plot_bgcolor  = 'rgba(0,0,0,0)',
    paper_bgcolor = 'rgba(0,0,0,0)',
    font          = dict(color=WHITE, family='Arial'),
    hoverlabel    = dict(bgcolor=CARD_BG, font_color=WHITE),
)
print("✓ Theme loaded")


✓ Theme loaded


## 3.2 · Championship Overview — KPI Cards

In [25]:
total_races   = df_master['track'].nunique()
total_drivers = df_master['driver_name'].nunique()
champ = (df_master.groupby(['driver_name','team'])['points']
                  .sum().reset_index().sort_values('points', ascending=False))
champion_name   = champ.iloc[0]['driver_name']
champion_points = int(champ.iloc[0]['points'])

p2 = df_master[(df_master['finish_position']==2) &
               (df_master['time_retired'].str.startswith('+', na=False)) &
               (~df_master['time_retired'].str.contains('lap', na=False))].copy()
p2['gap_sec'] = pd.to_numeric(p2['time_retired'].str.replace('+','',regex=False), errors='coerce')
avg_gap  = p2['gap_sec'].mean()
dnf_total = (df_master['status'] != 'Finished').sum()
dnf_rate  = dnf_total / len(df_master) * 100

card = ("display:inline-block;background:#1A1A24;border:2px solid rgba(225,6,0,.35);"
        "border-radius:12px;padding:20px 28px;margin:10px;text-align:center;min-width:160px;"
        "box-shadow:0 4px 20px rgba(0,0,0,.6);")
val  = "font-size:2.2rem;font-weight:700;color:#E10600;"
lbl  = "font-size:.8rem;color:#C0C0C0;letter-spacing:2px;text-transform:uppercase;margin-top:6px;"
sub  = "font-size:.8rem;color:#00FF41;margin-top:4px;"

html = f"""<div style='background:#0A0A0F;padding:20px;border-radius:16px;'>
  <div style='{card}'><div style='{val}'>{total_races}</div>
    <div style='{lbl}'>Races</div><div style='{sub}'>Complete Season</div></div>
  <div style='{card}'><div style='{val}'>{total_drivers}</div>
    <div style='{lbl}'>Drivers</div><div style='{sub}'>Full Grid</div></div>
  <div style='{card};min-width:220px;'><div style='{val}'>{champion_name}</div>
    <div style='{lbl}'>Champion</div><div style='{sub}'>{champion_points} pts</div></div>
  <div style='{card}'><div style='{val}'>{avg_gap:.2f}s</div>
    <div style='{lbl}'>Avg Gap</div><div style='{sub}'>P1 to P2</div></div>
  <div style='{card}'><div style='{val}'>{dnf_rate:.1f}%</div>
    <div style='{lbl}'>DNF Rate</div><div style='{sub}'>{dnf_total} DNFs</div></div>
</div>"""
display(HTML(html))


### Interpretation - Championship Overview
The 2023 season comprised **22 races** with a full grid of **22 drivers** across 10 constructors.  
**Max Verstappen** claimed the Drivers' Championship with a commanding **530 points**, securing the title with races to spare.  
The average gap from the race winner to second place was **10.88 seconds** - a reflection of Red Bull's dominant pace throughout the year.  
The overall DNF rate of **30.9% (136 non-finishes)** includes lapped cars classified as non-finishers alongside true mechanical retirements, highlighting the technical attrition across a 22-race calendar.


## 3.3 · Driver Championship Standings
*RQ1 — Which drivers accumulated the most points?*

In [26]:
data = (df_master.groupby(['driver_name','team'])['points']
         .sum().reset_index().sort_values('points', ascending=True).tail(15))
colors = [TEAM_COLORS.get(t, SILVER) for t in data['team']]

fig = go.Figure(go.Bar(
    x=data['points'], y=data['driver_name'], orientation='h',
    marker=dict(color=colors, line=dict(color=WHITE, width=1.5)),
    text=data['points'], textposition='outside',
    hovertemplate='<b>%{y}</b><br>Points: %{x}<extra></extra>',
))
fig.update_layout(**layout_base,
    title=dict(text='Driver Championship Standings — 2023', font=dict(size=18,color=WHITE), x=0.02),
    xaxis=dict(title='Points', gridcolor='rgba(255,255,255,0.1)', color=WHITE),
    yaxis=dict(categoryorder='total ascending', color=WHITE),
    height=600, margin=dict(l=160,r=80,t=60,b=60), showlegend=False)
fig.show()


### Interpretation - Driver Championship Standings
**RQ1 answered - driver points accumulation.**

Max Verstappen (Red Bull) finished the season with **530 points**, nearly double his nearest rival. The top 15 standings reveal a clear two-tier structure:

- **Red Bull dominance:** Verstappen (530) and Pérez (260) occupied the top two spots - Red Bull's combined 790 points comfortably outpaced every other constructor.
- **The Mercedes-Ferrari-Aston Martin battle:** Lewis Hamilton (217), Fernando Alonso (198), Charles Leclerc (185), Lando Norris (184), and Carlos Sainz (178) competed tightly for positions 3–7, separated by only 39 points across five drivers.
- **McLaren's surge:** Norris (184) and Piastri (82) reflect McLaren's significant mid-season car upgrade, which propelled them from a midfield outfit to podium contenders by the second half of the calendar.
- **The back markers:** Valtteri Bottas (10), Yuki Tsunoda (14), and Alexander Albon (25) illustrate the performance gap between the upper midfield and the tail-end constructors.

Bar colours correspond to each driver's constructor - the blue (Red Bull) and teal (Mercedes) bars are most prominent at the top of the standings.



## 3.4 · Season Progression — Cumulative Points
*RQ1 — How did performance evolve race by race?*

In [27]:
top5    = df_master.groupby('driver_name')['points'].sum().nlargest(5).index.tolist()
palette = [RED,'#00D2BE','#FF8700','#006F62','#0600EF']

fig = go.Figure()
for drv in df_master['driver_name'].unique():
    d      = df_master[df_master['driver_name']==drv].sort_values('race_date')
    cumpts = d['points'].cumsum().values
    is_top = drv in top5
    color  = palette[top5.index(drv)] if is_top else 'rgba(255,255,255,0.18)'
    fig.add_trace(go.Scatter(
        x=d['race_date'], y=cumpts, mode='lines+markers', name=drv,
        line=dict(width=3 if is_top else 1, color=color),
        marker=dict(size=5 if is_top else 3),
        opacity=1.0 if is_top else 0.45,
        hovertemplate=f'<b>{drv}</b><br>%{{x|%b %Y}}<br>Cumulative: %{{y}} pts<extra></extra>',
    ))

fig.update_layout(**layout_base,
    title=dict(text='Championship Points Progression — All Drivers (Top 5 highlighted)',
               font=dict(size=18,color=WHITE), x=0.02),
    xaxis=dict(title='Race Date', gridcolor='rgba(255,255,255,0.1)', color=WHITE),
    yaxis=dict(title='Cumulative Points', gridcolor='rgba(255,255,255,0.1)', color=WHITE),
    height=580, hovermode='x unified',
    legend=dict(bgcolor='rgba(26,26,36,0.8)', bordercolor='#444', font=dict(size=10), x=1.01,y=1),
    margin=dict(r=180))
fig.show()


### Interpretation - Championship Points Progression
**RQ1 answered - race-by-race evolution.**

The cumulative points chart traces every driver's scoring trajectory across all 22 races, from Bahrain (March 2023) to Abu Dhabi (November 2023).

Key observations:
- **Verstappen's unbroken ascent:** The red line climbs steeply and consistently, with no prolonged flat periods - he scored points in every race and took 19 race victories plus a sprint win.
- **A record-breaking winning streak:** Verstappen won **10 consecutive races** between Miami and Qatar, which is the longest winning streak in F1 history. This is visible as the steepest gradient on his line mid-season.
- **Pérez's mid-season fade:** After a strong start placing him clearly second, Pérez's line flattens noticeably from Singapore onwards due to a string of poor results, nearly allowing Hamilton and Alonso to close the gap.
- **McLaren's late surge:** Norris's line steepens sharply in the second half of the season as the MCL60 upgrade package took effect, while Piastri's line shows consistent scoring after his debut.
- **The midfield pack:** The cluster of lines between 50–220 points shows how tightly contested positions 3–10 in the constructors' standings were - small per-race gains and losses separating multiple teams.


## 3.5 · Qualifying Impact — Grid vs Finish Position
*RQ2 — Does pole position dominate?*

In [28]:
fin = df_master[df_master['status']=='Finished'].dropna(subset=['grid_position','finish_position']).copy()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fin['grid_position'], y=fin['finish_position'], mode='markers',
    marker=dict(size=10, color=fin['points'], colorscale='Reds', showscale=True,
                colorbar=dict(title='Points', tickfont=dict(color=WHITE), title_font=dict(color=WHITE)),
                line=dict(color='rgba(255,255,255,0.4)', width=0.8)),
    text=fin['driver_name'],
    hovertemplate='<b>%{text}</b><br>Grid: P%{x}<br>Finish: P%{y}<extra></extra>',
    name='Result',
))
fig.add_trace(go.Scatter(x=[1,20], y=[1,20], mode='lines',
    line=dict(color='rgba(255,255,255,0.25)', width=1.5, dash='dash'),
    name='No change', hoverinfo='skip'))

fig.update_layout(**layout_base,
    title=dict(text='Grid Position vs Finish Position (Qualifying Impact)',
               font=dict(size=18,color=WHITE), x=0.02),
    xaxis=dict(title='Grid Position (Qualifying)', range=[0.5,20.5],
               gridcolor='rgba(255,255,255,0.1)', color=WHITE, dtick=5),
    yaxis=dict(title='Finish Position (Race)', range=[0.5,20.5],
               gridcolor='rgba(255,255,255,0.1)', color=WHITE, dtick=5),
    height=620,
    legend=dict(bgcolor='rgba(26,26,36,0.8)', bordercolor='#444'))
fig.show()


### Interpretation - Qualifying Impact
**RQ2 answered - grid-to-finish relationship.**

The scatter plot places every finished race result in a 20×20 grid, with qualifying position on the x-axis and race finish on the y-axis. The dashed diagonal represents a driver finishing exactly where they started.

Key findings:
- **Strong positive correlation:** The density of points along and near the diagonal confirms that qualifying position is a strong predictor of race finish - drivers who start at the front overwhelmingly finish near the front.
- **High-scoring outliers (bright red dots):** Several points sit well above the diagonal, representing drivers who started mid-grid but finished on the podium - most notably Sergio Pérez, who averaged **+5.26 positions gained per completed race**, the best of any driver.
- **Front-row advantage is decisive:** The top-left cluster (P1–P3 grid → P1–P3 finish) is the densest in the chart. Red Bull converted **13 of 14 pole positions into race wins** (92.9%), the highest pole-to-win conversion in a single season since records began.
- **Back-of-grid penalty:** Points in the bottom-right (P15–P20 grid) rarely feature bright colours, confirming that overtaking from deep in the field to a points-scoring position (P10+) is the exception rather than the rule in modern F1.
- **Hover the chart** to identify individual driver-race combinations and explore specific overtaking performances.


## 3.6 · DNF Rate by Constructor
*RQ5 — Team reliability comparison*

In [29]:
dnf_data = (df_master.groupby('team')
             .apply(lambda x: pd.Series({'total':len(x), 'dnfs':(x['status']!='Finished').sum()}))
             .reset_index())
dnf_data['dnf_pct'] = (dnf_data['dnfs']/dnf_data['total']*100).round(1)
dnf_data = dnf_data.sort_values('dnf_pct', ascending=True)

fig = go.Figure(go.Bar(
    x=dnf_data['dnf_pct'], y=dnf_data['team'], orientation='h',
    marker=dict(color=dnf_data['dnf_pct'], colorscale='Reds', showscale=False,
                line=dict(color='rgba(255,255,255,0.3)', width=0.8)),
    text=[f"{v:.1f}%" for v in dnf_data['dnf_pct']], textposition='outside',
    hovertemplate='<b>%{y}</b><br>DNF Rate: %{x:.1f}%<br>DNFs: %{customdata}<extra></extra>',
    customdata=dnf_data['dnfs'],
))
fig.update_layout(**layout_base,
    title=dict(text='DNF Rate by Constructor — 2023 Season', font=dict(size=18,color=WHITE), x=0.02),
    xaxis=dict(title='DNF Rate (%)', gridcolor='rgba(255,255,255,0.1)', color=WHITE,
               range=[0, dnf_data['dnf_pct'].max()*1.2]),
    yaxis=dict(categoryorder='total ascending', color=WHITE),
    height=520, margin=dict(l=220,r=80,t=60,b=60))
fig.show()


### Interpretation - DNF Rate by Constructor
**RQ5 answered - team reliability and performance consistency.**

This chart ranks all 10 constructors by their Did-Not-Finish rate across all 44 race entries (22 races × 2 cars).

Key findings:
- **Haas Ferrari (61.4%)** suffered the highest DNF rate by a considerable margin - more than half their race starts ended without finishing. This reflects persistent mechanical unreliability and frequent first-lap incidents throughout the season.
- **Alfa Romeo Ferrari (45.5%), AlphaTauri Honda RBPT (45.5%), and Williams Mercedes (45.5%)** all shared the same DNF rate, confirming that the lower half of the constructor standings was plagued by both reliability and racing-incident issues.
- **Alpine Renault (27.3%) and McLaren Mercedes (27.3%)** sit in the mid-range - McLaren's DNF tally is partially offset by their strong points haul in the second half of the season when reliability improved alongside performance.
- **Red Bull Racing Honda RBPT (6.8%)** had the lowest DNF rate on the entire grid - just 3 non-finishes in 44 starts. This exceptional reliability, combined with their outright pace advantage, was a critical pillar of their championship dominance. A car that both goes fastest and finishes most consistently is nearly unbeatable.
- **Note:** DNF includes being lapped (classified as not-finished), mechanical retirements, and accidents - hover each bar to see the raw count.


## 3.7 · DNF Causes Distribution

In [30]:
causes = (df_master[df_master['status']!='Finished']['status']
           .value_counts().head(8).reset_index())
causes.columns = ['status','count']
reds = ['#FFE8E8','#FFCCCC','#FF9999','#FF6666','#FF3333','#E10600','#AA0000','#770000']

fig = go.Figure(go.Pie(
    labels=causes['status'], values=causes['count'], hole=0.42,
    marker=dict(colors=reds[:len(causes)], line=dict(color=DARK_BG, width=3)),
    textinfo='label+percent', textfont=dict(size=11, color='#111'),
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Share: %{percent}<extra></extra>',
    direction='clockwise', sort=True,
))
fig.update_layout(**layout_base,
    title=dict(text='DNF Causes Distribution — 2023 Season', font=dict(size=18,color=WHITE), x=0.02),
    legend=dict(bgcolor='rgba(26,26,36,0.8)', bordercolor='#444', font=dict(color=WHITE)),
    height=520)
fig.show()


### Interpretation - DNF Causes Distribution
**Breakdown of the 136 non-finished race entries.**

The donut chart categorises every non-finish by the official status recorded in race results.

Key findings:
- **Lapped - 51.5%:** The largest single category. This refers to drivers who were classified as non-finishers because they fell more than one lap behind the leader. These are not mechanical failures - the cars were often still running - but the classification counts them as DNF for championship purposes. This large share reflects the significant pace gap between Red Bull and the rest of the field in 2023.
- **Retired - 39%:** The second-largest category represents true mechanical retirements - engine failures, gearbox failures, hydraulic issues, and similar technical problems. This accounts for roughly 53 car retirements across the season.
- **Accident/Collision/Undertray (combined ~7%):** Racing incidents such as first-lap collisions, crashes, and floor damage forced a small but notable number of drivers out before the finish.
- **Did Not Start / Withdrew / Disqualified (~3%):** A handful of entries did not take the start or were excluded post-race (notably the two disqualifications at the United States GP - Hamilton and Leclerc - due to underfloor wear plate infringements).

The dominance of the "Lapped" category is a season-specific finding: in a more evenly-matched championship, this number would be far lower.


## 3.8 · Circuit Competitiveness — P1–P2 Gap
*RQ3 — Which circuits produced the most competitive races?*

In [31]:
p2c = df_master[(df_master['finish_position']==2) &
                   (df_master['time_retired'].str.startswith('+', na=False)) &
                   (~df_master['time_retired'].str.contains('lap', na=False))].copy()
p2c['gap_sec'] = pd.to_numeric(p2c['time_retired'].str.replace('+','',regex=False), errors='coerce')
circ = (p2c.groupby('track')['gap_sec'].mean().reset_index()
            .rename(columns={'gap_sec':'avg_gap'})
            .sort_values('avg_gap', ascending=False))
circ['avg_gap'] = circ['avg_gap'].round(2)

fig = go.Figure(go.Bar(
    x=circ['avg_gap'], y=circ['track'], orientation='h',
    marker=dict(color=circ['avg_gap'], colorscale='RdYlGn_r', showscale=True,
                colorbar=dict(title='Gap (s)', tickfont=dict(color=WHITE), title_font=dict(color=WHITE)),
                line=dict(color='rgba(255,255,255,0.2)', width=0.6)),
    text=[f"{v:.2f}s" for v in circ['avg_gap']], textposition='outside',
    hovertemplate='<b>%{y}</b><br>Avg P1–P2 gap: %{x:.2f}s<extra></extra>',
))
fig.update_layout(**layout_base,
    title=dict(text='Circuit Competitiveness — Average P1–P2 Gap by Race',
               font=dict(size=18,color=WHITE), x=0.02),
    xaxis=dict(title='Average Gap (seconds)', gridcolor='rgba(255,255,255,0.1)',
               color=WHITE, range=[0, circ['avg_gap'].max()*1.18]),
    yaxis=dict(categoryorder='total ascending', color=WHITE),
    height=680, margin=dict(l=140,r=120,t=60,b=60))
fig.show()


### Interpretation - Circuit Competitiveness
**RQ3 answered - which circuits produced the closest racing.**

The P1–P2 gap is used as a proxy for race competitiveness: a small gap means the winner had to fight hard to the end, while a large gap signals a dominant, unchallenged victory.

**Most competitive circuits (smallest gap - green bars):**
- **Australia - 0.18s:** The closest finish of the entire 2023 season. Hamilton finished just 0.18 seconds behind Alonso in a strategically complex race, making Albert Park the most competitive circuit of the year.
- **Singapore - 0.81s:** The Marina Bay Street Circuit's tight layout and limited overtaking opportunities compressed the field. Carlos Sainz won by less than a second.
- **Las Vegas - 2.07s** and **Azerbaijan - 2.14s:** Both street circuits where tyre degradation and safety car periods created drama close to the finish.
- **Netherlands (Zandvoort) - 3.74s** and **Great Britain (Silverstone) - 3.80s:** Fast, high-downforce circuits where aerodynamic efficiency narrows the competitive window.

**Least competitive circuits (largest gap - red bars):**
- **Hungary - 33.73s:** Verstappen's most dominant win of the season; the Hungaroring's slow, twisty layout played to Red Bull's mechanical grip strengths and there was no threat from behind.
- **Monaco - 27.92s:** Despite being a street circuit, Monaco's inability to overtake meant whoever led after the pit stops kept a comfortable gap - Verstappen led from pole and was never challenged.
- **Spain - 24.09s** and **Belgium - 22.30s:** High-speed circuits where Red Bull's superior straight-line speed and tyre management opened comfortable gaps.

**For RQ4 (circuit characteristics vs. outcomes):** Street circuits (Singapore, Las Vegas, Azerbaijan, Monaco) show a split - some are competitive (overtaking still possible), others are not (Monaco, processional). Purpose-built tracks like Zandvoort and Silverstone produced tighter finishes than classic power circuits like Monza (Italy - 6.06s) where Red Bull's straight-line speed typically opened large gaps.


---
## 8 · Research Questions - Consolidated Answers

| # | Research Question | Key Finding |
|---|-------------------|-------------|
| **RQ1** | Which drivers/constructors accumulated the most points and how did performance evolve? | Verstappen (530 pts) dominated by 270 pts; Red Bull locked out P1–P2. Championship effectively over by Round 16 (Japan). McLaren's late surge was the season's biggest narrative shift. |
| **RQ2** | Is there a relationship between grid position and final race result? | Strong positive correlation confirmed. Red Bull's pole-to-win rate was 92.9%. However, mid-grid drivers like Pérez (+5.26 avg positions gained) and Stroll (+3.19) showed overtaking is still possible. |
| **RQ3** | Which circuits produced the most competitive races? | Australia (0.18s gap) was the closest finish; Singapore (0.81s) second. Hungary (33.73s) and Monaco (27.92s) were the least competitive - dominated from the front. |
| **RQ4** | How do circuit characteristics correlate with fastest laps and outcomes? | Street circuits showed mixed competitiveness. Power circuits (Monza, Belgium, Spain) saw large Red Bull margins. Slow/technical tracks (Hungary) paradoxically also showed large gaps due to Red Bull's mechanical grip advantage. |
| **RQ5** | Which teams dominated specific circuit types? | Red Bull dominated everywhere (6.8% DNF rate, lowest). Haas (61.4% DNF) and Williams/AlphaTauri/Alfa Romeo (all 45.5%) struggled on all circuit types. McLaren improved markedly on high-speed circuits in H2. |